In [ ]:
import os
import io
import time
import logging
import pandas as pd
import ccxt

# ═══════════════════════════════════════════════════════════
#  LIVE INITIALIZATION & CONFIGURATION CREDENTIALS
# ═══════════════════════════════════════════════════════════
PARAMS = {
    "API_KEY": "YOUR_BINANCE_TESTNET_API_KEY",      # Replace with Testnet Key
    "API_SECRET": "YOUR_BINANCE_TESTNET_SECRET",    # Replace with Testnet Secret
    "TRADE_SIZE_USDT": 100,                         # Allocated size per arb attempt ($)
    "MIN_PROFIT_TRIGGER": 1.001,                    # Minimum gross multiple to fire orders (0.1% gross)
    "USE_PROXIES": False,                           # Flip to True if your ISP blocks Binance APIs
    "PROXY_ADDRESS": "http://127.0.0.1:7890",       # Local network proxy listener port
    "EXCEL_FILE": "live_arb_execution_log.xlsx"
}

# Define execution path tracking keys
TRIANGLE_PATH = {
    "base": "USDT", "b": "BTC", "c": "ETH",
    "pair1": "BTC/USDT", "pair2": "ETH/BTC", "pair3": "ETH/USDT"
}

log_capture_string = io.StringIO()
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(), logging.StreamHandler(log_capture_string)]
)
log = logging.getLogger("LiveArbEngine")

# ═══════════════════════════════════════════════════════════
#  EXCHANGE INITIALIZATION LAYER
# ═══════════════════════════════════════════════════════════
def init_live_exchange():
    config = {
        'apiKey': PARAMS["API_KEY"],
        'secret': PARAMS["API_SECRET"],
        'enableRateLimit': True,
        'options': {'defaultType': 'spot'}
    }
    
    # Inject secure proxy routing values to bypass regional DNS blocking structures safely
    if PARAMS["USE_PROXIES"]:
        config['proxies'] = {
            'http': PARAMS["PROXY_ADDRESS"],
            'https': PARAMS["PROXY_ADDRESS"]
        }
        log.info(f"Routing data channels via localized proxy pipeline: {PARAMS['PROXY_ADDRESS']}")

    exchange = ccxt.binance(config)
    
    # FORCE SANDBOX MODE to point to the safe Testnet cloud matching system
    exchange.set_sandbox_mode(True)
    log.info("Exchange initialized successfully: CONNECTED TO BINANCE SPOT TESTNET")
    return exchange

# ═══════════════════════════════════════════════════════════
#  LIVE EXECUTION ROUTER ENGINE
# ═══════════════════════════════════════════════════════════
def execute_live_arbitrage_loop():
    exchange = init_live_exchange()
    order_records = []
    
    try:
        log.info("Synchronizing core asset symbol structures and wallet matrices...")
        exchange.load_markets()
        balances = exchange.fetch_balance()
        log.info(f"Wallet Connected. Current Available USDT Balance: ${balances['free'].get('USDT', 0.0):,.2f}")
    except Exception as e:
        log.critical(f"System boot failed during initialization phase: {e}")
        return

    # Keep scanning until manually stopped (Ctrl+C)
    cycle = 0
    while True:
        cycle += 1
        try:
            # 1. Fetch real-time tickers for the triangular circuit
            symbols = [TRIANGLE_PATH["pair1"], TRIANGLE_PATH["pair2"], TRIANGLE_PATH["pair3"]]
            tickers = exchange.fetch_tickers(symbols)
            
            # Step 1: Buy BTC with USDT (Pay current Ask)
            p1_ask = tickers[TRIANGLE_PATH["pair1"]]['ask']
            # Step 2: Swap BTC into ETH (Pay current Ask on ETH/BTC cross)
            p2_ask = tickers[TRIANGLE_PATH["pair2"]]['ask']
            # Step 3: Cash out ETH back to USDT (Receive current Bid)
            p3_bid = tickers[TRIANGLE_PATH["pair3"]]['bid']
            
            if not (p1_ask and p2_ask and p3_bid):
                continue
                
            # 2. Math model calculation (simulating execution path returns)
            gross_return_multiple = (1.0 / p1_ask) * (1.0 / p2_ask) * p3_bid
            
            if cycle % 10 == 0:
                log.info(f"Scan Loop #{cycle} | Implied Gross Arbitrage Return Multiple: {gross_return_multiple:.6f}")
                
            # 3. Live Order Fire Condition Trigger
            if gross_return_multiple >= PARAMS["MIN_PROFIT_TRIGGER"]:
                log.info(f"💥 EXCISED DISCREPANCY! Projected Gross Factor: {gross_return_multiple:.6f}. Executing sequential orders...")
                
                # Dynamic order calculation based on parameters
                amount_b = PARAMS["TRADE_SIZE_USDT"] / p1_ask
                
                # --- LEG 1: Buy Asset B (BTC/USDT) ---
                log.info(f"[LEG 1] Firing Market Buy Order for {TRIANGLE_PATH['pair1']} | Size: {amount_b:.6f}")
                order1 = exchange.create_market_buy_order(TRIANGLE_PATH["pair1"], amount_b)
                filled_b = order1['filled'] if 'filled' in order1 else amount_b
                
                # --- LEG 2: Cross Swap into C (ETH/BTC) ---
                # Calculating amount of C to buy based on the filled balance of B
                amount_c = filled_b / p2_ask
                log.info(f"[LEG 2] Firing Market Buy Order for {TRIANGLE_PATH['pair2']} | Size: {amount_c:.6f}")
                order2 = exchange.create_market_buy_order(TRIANGLE_PATH["pair2"], amount_c)
                filled_c = order2['filled'] if 'filled' in order2 else amount_c
                
                # --- LEG 3: Sell back to Base Currency (ETH/USDT) ---
                log.info(f"[LEG 3] Firing Market Sell Order for {TRIANGLE_PATH['pair3']} | Size: {filled_c:.6f}")
                order3 = exchange.create_market_sell_order(TRIANGLE_PATH["pair3"], filled_c)
                final_received_usdt = order3['cost'] if 'cost' in order3 else (filled_c * p3_bid)
                
                net_profit_loss = final_received_usdt - PARAMS["TRADE_SIZE_USDT"]
                log.info(f"🎉 Circuit completed! Final Realized Trade Net Return PnL: ${net_profit_loss:+.4f}")
                
                order_records.append({
                    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "Cycle": cycle,
                    "Gross Expected Multiple": round(gross_return_multiple, 6),
                    "Net PnL Realized ($)": round(net_profit_loss, 4),
                    "Order 1 ID": order1.get('id', 'N/A'),
                    "Order 2 ID": order2.get('id', 'N/A'),
                    "Order 3 ID": order3.get('id', 'N/A')
                })
                
                # Brief execution rest interval to let exchange matching states settle down
                time.sleep(5)
                
        except ccxt.NetworkError as ne:
            log.warning(f"Temporary exchange connectivity error experienced: {ne}")
            time.sleep(2)
        except ccxt.InsufficientFunds as ie:
            log.error(f"Execution halted. Wallet lacks sufficient assets to trade: {ie}")
            break
        except KeyboardInterrupt:
            log.info("System shut down command intercepted. Exporting audit data trails...")
            break
        except Exception as e:
            log.error(f"Unexpected operational loop error: {e}")
            time.sleep(2)
            
        time.sleep(0.5) # Fast-polling scanning pacing interval delay

    # ═══════════════════════════════════════════════════════════
    #  EXCEL AUDIT SHEET WRITE PIPELINE
    # ═══════════════════════════════════════════════════════════
    if order_records:
        summary_df = pd.DataFrame(order_records)
    else:
        summary_df = pd.DataFrame(columns=["Timestamp", "Cycle", "Gross Expected Multiple", "Net PnL Realized ($)"])
        
    param_df = pd.DataFrame(list(PARAMS.items()), columns=["System Operational Setting Key", "Configured Param Value"])
    log_df = pd.DataFrame(log_capture_string.getvalue().split('\n'), columns=["System Live Running Node Output Logs"])
    
    with pd.ExcelWriter(PARAMS["EXCEL_FILE"], engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Live_Trade_Ledger", index=False)
        param_df.to_excel(writer, sheet_name="Bot_Settings_Audit", index=False)
        log_df.to_excel(writer, sheet_name="Terminal_Logs", index=False)
        
    print("\n" + "═"*65)
    print(" LIVE SESSION HALTED — WORKBOOK LEDGERS ARCHIVED")
    print("═"*65)
    print(f" Master Sheet compiled inside local directory path: {PARAMS['EXCEL_FILE']}")
    print("═"*65)

if __name__ == "__main__":
    execute_live_arbitrage_loop()

13:28:01 [INFO] Exchange initialized successfully: CONNECTED TO BINANCE SPOT TESTNET
13:28:01 [INFO] Synchronizing core asset symbol structures and wallet matrices...
13:28:07 [CRITICAL] System boot failed during initialization phase: binance {"code":-2014,"msg":"API-key format invalid."}
